# GameTheory-3 : Topologie des Jeux 2×2 — Twin C# (classification ordinale from-scratch)

**Twin C# (.NET Interactive)** de [GameTheory-3-Topology2x2](GameTheory-3-Topology2x2.ipynb) — marathon **#4956** (parité .NET ⇄ Python), volet **GameTheory topologie ordinale**, axe-2 SOTA **#3801** Prong B. Suite directe de [GameTheory-2-NormalForm-Csharp](GameTheory-2-NormalForm-Csharp.ipynb).

Le notebook Python utilise **networkx** (graphe des swaps), **numpy** (matrices) et **matplotlib** (visualisation) pour classifier les 576 jeux 2×2 ordinaux selon leur topologie d'équilibres. Ce twin **déroule toute la combinatoire from-scratch** (BCL .NET 9, **0 NuGet**) : permutations ordinales, swaps de rangs, **BFS shortest-path** sur le graphe des swaps (au lieu de networkx), recherche des équilibres de Nash purs (best-response), classification.

## Objectifs d'apprentissage

1. Représenter un jeu 2×2 en **gain ordinaux** (rangs 1-4, indépendants des valeurs cardinales).
2. Énumérer les **576 jeux** (24×24 permutations) et comprendre la structure de l'espace.
3. Définir la **topologie des swaps** (transformations élémentaires entre jeux).
4. Calculer les **distances** entre jeux classiques (BFS sur le graphe des swaps).
5. Classer par **structure d'équilibres de Nash** (0, 1, 2+ équilibres purs).


## Introduction

### Pourquoi la représentation ordinale ?

Beaucoup de propriétés stratégiques d'un jeu 2×2 ne dépendent que de l'**ordre** des gains, pas de leurs valeurs numériques. Le Dilemme du Prisonnier reste un dilemme que les gains soient (3,1,4,2) ou (30,10,40,20) : seule compte l'ordre **Temptation > Reward > Punishment > Sucker**. La représentation ordinale (rangs 1-4) capture cette invariance.

### L'espace des jeux

Chaque joueur a 4 cellules (2×2) et ses gains forment une **permutation de (1,2,3,4)**. Il y a $4! = 24$ permutations par joueur, donc $24 \times 24 = \mathbf{576}$ jeux ordinaux distincts. C'est un espace fini qu'on peut explorer exhaustivement.

### Topologie des swaps

Deux jeux sont **voisins** s'ils diffèrent d'un seul **swap adjacent** (échanger deux rangs consécutifs 1↔2, 2↔3, ou 3↔4) pour un joueur. Le graphe des swaps connecte les 576 jeux : la **distance** entre deux jeux = nombre minimal de swaps. Cette métrique révèle quelles familles de jeux sont « proches » stratégiquement.

### Complémentarité (#3801 Prong B)

| Aspect | Python (twin) | Twin C# (ici) |
|--------|---------------|----------------|
| Graphe des swaps | `networkx.Graph` + BFS | **BFS from-scratch (Dictionary + Queue)** |
| Matrices | `numpy.ndarray` | **tableaux 2D `int[,]`** |
| Visualisation | `matplotlib` (labyrinthe coloré) | **tables console + ASCII** |
| Énumération | `itertools.permutations` | **Heap's algorithm from-scratch** |
| Dépendances | networkx, numpy, matplotlib | **BCL seule, 0 NuGet** |


## 1. Représentation ordinale — `OrdinalGame`

Un jeu 2×2 ordinal = deux tuples de 4 rangs (un par joueur). Convention de numérotation des cellules :
```
  0 | 1       (Ligne=Haut/Bas, Colonne=Gauche/Droite)
  -----
  2 | 3
```
Chaque tuple doit être une **permutation de (1,2,3,4)** — vérifié à la construction.


In [1]:
// Cellule 1 — OrdinalGame (persistant cross-cell via kernel .net-csharp)
using System;
using System.Collections.Generic;
using System.Linq;

public sealed class OrdinalGame : IEquatable<OrdinalGame>
{
    public int[] Row { get; }   // 4 rangs du joueur Ligne (cellules 0,1,2,3)
    public int[] Col { get; }   // 4 rangs du joueur Colonne
    public string Name { get; }

    public OrdinalGame(IEnumerable<int> row, IEnumerable<int> col, string name = "")
    {
        Row = row.ToArray(); Col = col.ToArray(); Name = name;
        if (Row.Length != 4 || Col.Length != 4)
            throw new ArgumentException("Les gains doivent avoir 4 cellules.");
        if (!IsPerm1234(Row) || !IsPerm1234(Col))
            throw new ArgumentException("Les gains doivent etre une permutation de (1,2,3,4).");
    }

    private static bool IsPerm1234(int[] a)
    {
        var s = a.OrderBy(x => x).ToArray();
        return s[0] == 1 && s[1] == 2 && s[2] == 3 && s[3] == 4;
    }

    // Matrice 2x2 du joueur Ligne (A[i,j]) et Colonne (B[i,j]).
    public int[,] RowMatrix() => new int[,] { { Row[0], Row[1] }, { Row[2], Row[3] } };
    public int[,] ColMatrix() => new int[,] { { Col[0], Col[1] }, { Col[2], Col[3] } };

    public override bool Equals(object o) => Equals(o as OrdinalGame);
    public bool Equals(OrdinalGame g) => g != null && Row.SequenceEqual(g.Row) && Col.SequenceEqual(g.Col);
    public override int GetHashCode() => string.Join(",", Row).GetHashCode() ^ (string.Join(",", Col).GetHashCode() << 1);
    public override string ToString() => $"Row[{string.Join(",", Row)}] Col[{string.Join(",", Col)}]";
}

display("OrdinalGame defini. Validation : permutation de (1,2,3,4) enforcee.");


The below script needs to be able to find the current output cell; this is an easy method to get it.

OrdinalGame defini. Validation : permutation de (1,2,3,4) enforcee.

## 2. Catalogue des jeux classiques

Les 5 archétypes stratégiques fondamentaux, en notation ordinale. Convention T>R>P>S (Temptation, Reward, Punishment, Sucker) pour le Dilemme du Prisonnier :
- **T=4** (défection face à coopération), **R=3** (coopération mutuelle), **P=2** (défection mutuelle), **S=1** (coopération face à défection).


In [2]:
// Cellule 2 — Catalogue des jeux classiques
var ClassicGames = new Dictionary<string, OrdinalGame>
{
    ["Prisoner's Dilemma"] = new OrdinalGame(new[]{3,1,4,2}, new[]{3,4,1,2}, "PD"),
    ["Stag Hunt"]          = new OrdinalGame(new[]{4,1,3,2}, new[]{4,3,1,2}, "Stag"),
    ["Battle of Sexes"]    = new OrdinalGame(new[]{4,1,2,3}, new[]{3,2,1,4}, "BoS"),
    ["Chicken"]            = new OrdinalGame(new[]{3,2,4,1}, new[]{3,4,2,1}, "Chicken"),
    ["Matching Pennies"]   = new OrdinalGame(new[]{4,2,3,1}, new[]{2,4,1,3}, "MP"),
};

var sb = new System.Text.StringBuilder();
sb.AppendLine(" Jeu                   | Row (TL,TR,BL,BR) | Col (TL,TR,BL,BR)");
sb.AppendLine(new string('-', 62));
foreach (var kv in ClassicGames)
{
    var g = kv.Value;
    sb.AppendLine($" {kv.Key,-21} | ({string.Join(",", g.Row)}) | ({string.Join(",", g.Col)})");
}
sb.ToString().Display();


 Jeu                   | Row (TL,TR,BL,BR) | Col (TL,TR,BL,BR)
--------------------------------------------------------------
 Prisoner's Dilemma    | (3,1,4,2) | (3,4,1,2)
 Stag Hunt             | (4,1,3,2) | (4,3,1,2)
 Battle of Sexes       | (4,1,2,3) | (3,2,1,4)
 Chicken               | (3,2,4,1) | (3,4,2,1)
 Matching Pennies      | (4,2,3,1) | (2,4,1,3)


## 3. Transformations élémentaires — swaps de rangs

Un **swap adjacent** échange deux rangs consécutifs (1↔2, 2↔3, 3↔4) dans les gains d'un seul joueur. Ce sont les **arêtes** du graphe des swaps. Appliquer un swap à un jeu donne un jeu voisin (toujours une permutation valide de 1-4).


In [3]:
// Cellule 3 — Swaps de rangs (transformations elementaires)
static int[] SwapRanks(int[] payoffs, int rank1, int rank2)
{
    var r = (int[])payoffs.Clone();
    for (int i = 0; i < r.Length; i++)
    {
        if (r[i] == rank1) r[i] = rank2;
        else if (r[i] == rank2) r[i] = rank1;
    }
    return r;
}

static OrdinalGame ApplyRowSwap(OrdinalGame g, int r1, int r2)
    => new OrdinalGame(SwapRanks(g.Row, r1, r2), g.Col, g.Name + $"_R{r1}{r2}");

static OrdinalGame ApplyColSwap(OrdinalGame g, int r1, int r2)
    => new OrdinalGame(g.Row, SwapRanks(g.Col, r1, r2), g.Name + $"_C{r1}{r2}");

// Demonstration : un swap sur le Dilemme du Prisonnier
var pd = ClassicGames["Prisoner's Dilemma"];
var pdR34 = ApplyRowSwap(pd, 3, 4);   // echange Reward<->Temptation cote Ligne
$"PD original          : {pd}".Display();
$"PD apres swap Row 3-4: {pdR34}  (Row devient {string.Join(",", pdR34.Row)})".Display();
"Les 3 swaps adjacents possibles : (1,2), (2,3), (3,4) — par joueur.".Display();


PD original          : Row[3,1,4,2] Col[3,4,1,2]

PD apres swap Row 3-4: Row[4,1,3,2] Col[3,4,1,2]  (Row devient 4,1,3,2)

Les 3 swaps adjacents possibles : (1,2), (2,3), (3,4) — par joueur.

## 4. Plus court chemin de swaps — BFS

Le graphe des swaps connecte les 576 jeux : chaque arête = un swap adjacent (Row ou Col). Trouver la **distance** entre deux jeux = BFS shortest-path. Le twin Python utilise `networkx.shortest_path_length` ; ici on implémente le BFS **from-scratch** (`Queue` + `HashSet` des visités).

**Voisinage** : depuis un jeu, 6 voisins (3 swaps × 2 joueurs). Le graphe a 576 nœuds et ~1728 arêtes.


In [4]:
// Cellule 4 — BFS shortest-path entre deux jeux (graphe des swaps)
static (int distance, List<string> path) SwapDistance(OrdinalGame start, OrdinalGame target)
{
    var swaps = new[] { (1,2), (2,3), (3,4) };
    var queue = new Queue<(OrdinalGame g, List<string> path)>();
    queue.Enqueue((start, new List<string>()));
    var visited = new HashSet<OrdinalGame> { start };

    while (queue.Count > 0)
    {
        var (cur, path) = queue.Dequeue();
        if (cur.Equals(target)) return (path.Count, path);
        foreach (var (r1, r2) in swaps)
        {
            var nbr = ApplyRowSwap(cur, r1, r2);
            if (visited.Add(nbr))
            {
                var np = new List<string>(path); np.Add($"R{r1}{r2}");
                queue.Enqueue((nbr, np));
            }
            var nbc = ApplyColSwap(cur, r1, r2);
            if (visited.Add(nbc))
            {
                var np = new List<string>(path); np.Add($"C{r1}{r2}");
                queue.Enqueue((nbc, np));
            }
        }
    }
    return (-1, null);   // unreachable (ne devrait pas arriver : graphe connexe)
}

// Distance PD -> chaque autre jeu classique
$"Distances depuis Prisoner's Dilemma :".Display();
foreach (var kv in ClassicGames)
{
    if (kv.Key == "Prisoner's Dilemma") continue;
    var (d, path) = SwapDistance(pd, kv.Value);
    var pathStr = path != null ? string.Join(" -> ", path) : "n/a";
    $"  PD -> {kv.Key,-18} : {d} swaps  [{pathStr}]".Display();
}


Distances depuis Prisoner's Dilemma :

  PD -> Stag Hunt          : 2 swaps  [R34 -> C34]

  PD -> Battle of Sexes    : 5 swaps  [C23 -> R34 -> R23 -> C34 -> C23]

  PD -> Chicken            : 2 swaps  [R12 -> C12]

  PD -> Matching Pennies   : 3 swaps  [R12 -> C23 -> R34]

## 5. Énumération exhaustive — les 576 jeux

Chaque joueur a $4! = 24$ permutations de (1,2,3,4). L'espace total = $24 \times 24 = 576$ jeux ordinaux. On les énumère par **Heap's algorithm** (génération efficace des permutations, from-scratch) puis on vérifie la connexité du graphe des swaps.


In [5]:
// Cellule 5 — Enumeration exhaustive (Heap's algorithm for permutations)
static List<int[]> Permutations1234()
{
    var res = new List<int[]>();
    void Heap(int[] a, int k)
    {
        if (k == 1) { res.Add((int[])a.Clone()); return; }
        for (int i = 0; i < k; i++)
        {
            Heap(a, k - 1);
            if (k % 2 == 0) (a[i], a[k-1]) = (a[k-1], a[i]);
            else             (a[0], a[k-1]) = (a[k-1], a[0]);
        }
    }
    Heap(new int[] { 1, 2, 3, 4 }, 4);
    return res;
}

var allPerms = Permutations1234();
var allGames = new List<OrdinalGame>();
foreach (var rp in allPerms)
    foreach (var cp in allPerms)
        allGames.Add(new OrdinalGame(rp, cp));

$"Permutations de (1,2,3,4) : {allPerms.Count} (attendu 4! = 24)".Display();
$"Jeux ordinaux totaux     : {allGames.Count} (attendu 24 x 24 = 576)".Display();
$"Jeux distincts (HashSet) : {allGames.Distinct().Count()}".Display();


Permutations de (1,2,3,4) : 24 (attendu 4! = 24)

Jeux ordinaux totaux     : 576 (attendu 24 x 24 = 576)

Jeux distincts (HashSet) : 576

## 6. Équilibres de Nash purs — best response

Un équilibre de Nash pur est une cellule $(i,j)$ où aucun joueur n'a intérêt à dévier unilatéralement : $A[i,j]$ est le max de sa colonne (Ligne) ET $B[i,j]$ est le max de sa ligne (Colonne). On les trouve par **best-response** check sur les 4 cellules.


In [6]:
// Cellule 6 — Equilibres de Nash purs (best response)
static List<(int i, int j)> FindPureNash(OrdinalGame g)
{
    var A = g.RowMatrix();   // A[i,j] = gain Ligne
    var B = g.ColMatrix();   // B[i,j] = gain Colonne
    var eq = new List<(int, int)>();
    for (int i = 0; i < 2; i++)
    for (int j = 0; j < 2; j++)
    {
        bool rowBR = A[i, j] >= A[1 - i, j];   // Ligne best-response a Colonne=j
        bool colBR = B[i, j] >= B[i, 1 - j];   // Colonne best-response a Ligne=i
        if (rowBR && colBR) eq.Add((i, j));
    }
    return eq;
}

string CellName(int i, int j) => (i == 0 ? "T" : "B") + (j == 0 ? "L" : "R");

"Nash purs par jeu classique :".Display();
var sb6 = new System.Text.StringBuilder();
sb6.AppendLine(" Jeu                   | #Nash | Cellules d'equilibre");
sb6.AppendLine(new string('-', 56));
foreach (var kv in ClassicGames)
{
    var eq = FindPureNash(kv.Value);
    var cells = eq.Count == 0 ? "(aucun)" : string.Join(", ", eq.Select(e => CellName(e.i, e.j)));
    sb6.AppendLine($" {kv.Key,-21} | {eq.Count,5} | {cells}");
}
sb6.ToString().Display();
"Note : ces 5 jeux classiques ont tous 1 ou 2 Nash purs. Les jeux SANS aucun Nash pur (cycles de best-response) existent mais ne sont pas dans ce catalogue — voir l'histogramme cellule 7 (72/576 = 12,5% des jeux ordinaux ont 0 Nash pur).".Display();


Nash purs par jeu classique :

 Jeu                   | #Nash | Cellules d'equilibre
--------------------------------------------------------
 Prisoner's Dilemma    |     1 | BR
 Stag Hunt             |     2 | TL, BR
 Battle of Sexes       |     2 | TL, BR
 Chicken               |     2 | TR, BL
 Matching Pennies      |     1 | TR


Note : ces 5 jeux classiques ont tous 1 ou 2 Nash purs. Les jeux SANS aucun Nash pur (cycles de best-response) existent mais ne sont pas dans ce catalogue — voir l'histogramme cellule 7 (72/576 = 12,5% des jeux ordinaux ont 0 Nash pur).

## 7. Classification par structure de Nash

Sur les 576 jeux, on compte combien ont 0, 1, 2, 3 ou 4 équilibres purs. Cette distribution révèle la **topologie de l'espace** : la plupart des jeux ont 1-2 équilibres, une minorité n'en a aucun (Matching Pennies et voisins).


In [7]:
// Cellule 7 — Classification des 576 jeux par nombre de Nash
var histogram = new Dictionary<int, int>();
foreach (var g in allGames)
{
    int n = FindPureNash(g).Count;
    if (!histogram.ContainsKey(n)) histogram[n] = 0;
    histogram[n]++;
}

var sb7 = new System.Text.StringBuilder();
sb7.AppendLine(" #Nash | #Jeux | %     | Histogramme");
sb7.AppendLine(new string('-', 50));
foreach (var kv in histogram.OrderBy(x => x.Key))
{
    double pct = 100.0 * kv.Value / allGames.Count;
    int bars = (int)Math.Round(pct);   // 1 char par %
    sb7.AppendLine($" {kv.Key,5} | {kv.Value,5} | {pct,4:F1}% | {new string('#', bars)}");
}
sb7.ToString().Display();

int totalChecked = histogram.Values.Sum();
$"Total verifie : {totalChecked}/576 (attendu 576)".Display();
int gamesWithNash = histogram.Where(x => x.Key >= 1).Sum(x => x.Value);
$"{gamesWithNash} jeux ont au moins 1 Nash pur ({100.0*gamesWithNash/allGames.Count:F1}%).".Display();


 #Nash | #Jeux | %     | Histogramme
--------------------------------------------------
     0 |    72 | 12,5% | ############
     1 |   432 | 75,0% | ###########################################################################
     2 |    72 | 12,5% | ############


Total verifie : 576/576 (attendu 576)

504 jeux ont au moins 1 Nash pur (87,5%).

## 8. Connexité du graphe des swaps

Le graphe des swaps (576 nœuds, arêtes = swaps adjacents) est-il **connexe** ? C'est-à-dire : peut-on transformer n'importe quel jeu en n'importe quel autre par une séquence de swaps ? On le vérifie par BFS depuis un nœud — si on atteint les 576, le graphe est connexe.


In [8]:
// Cellule 8 — Connexite du graphe des swaps (BFS flood-fill depuis 1 noeud)
static int ReachableCount(OrdinalGame start)
{
    var swaps = new[] { (1,2), (2,3), (3,4) };
    var queue = new Queue<OrdinalGame>();
    queue.Enqueue(start);
    var visited = new HashSet<OrdinalGame> { start };
    while (queue.Count > 0)
    {
        var cur = queue.Dequeue();
        foreach (var (r1, r2) in swaps)
        {
            var nbr = ApplyRowSwap(cur, r1, r2);
            if (visited.Add(nbr)) queue.Enqueue(nbr);
            var nbc = ApplyColSwap(cur, r1, r2);
            if (visited.Add(nbc)) queue.Enqueue(nbc);
        }
    }
    return visited.Count;
}

int reachable = ReachableCount(pd);
$"BFS flood-fill depuis PD : {reachable} jeux atteignables / 576".Display();
display(reachable == 576
    ? "[OK] Graphe des swaps CONNEXE : tout jeu est transformable en tout autre par swaps."
    : $"[WARN] Graphe NON connexe : {reachable}/576 atteignables (composantes disjointes).");


BFS flood-fill depuis PD : 576 jeux atteignables / 576

[OK] Graphe des swaps CONNEXE : tout jeu est transformable en tout autre par swaps.

## 9. Matrice des distances entre jeux classiques

Tableau croisé : distance (en nombre de swaps) entre chaque paire de jeux classiques. Les jeux « proches » partagent une structure stratégique similaire ; les jeux « éloignés » (ex. Matching Pennies vs Stag Hunt) diffèrent radicalement.


In [9]:
// Cellule 9 — Matrice des distances entre jeux classiques (BFS)
var names = ClassicGames.Keys.ToList();
var sb9 = new System.Text.StringBuilder();
sb9.Append("                  ");
foreach (var n in names) sb9.Append($"{n.Substring(0, Math.Min(8, n.Length)),8} ");
sb9.AppendLine();
sb9.AppendLine(new string('-', 14 + names.Count * 9));
foreach (var n1 in names)
{
    sb9.Append($"{n1.Substring(0, Math.Min(13, n1.Length)),-13} ");
    foreach (var n2 in names)
    {
        int d = (n1 == n2) ? 0 : SwapDistance(ClassicGames[n1], ClassicGames[n2]).distance;
        sb9.Append($"{d,8} ");
    }
    sb9.AppendLine();
}
"Matrice des distances (nombre de swaps) entre jeux classiques :".Display();
sb9.ToString().Display();
"La diagonale = 0 (meme jeu). Symetrie par construction (graphe non oriente).".Display();


Matrice des distances (nombre de swaps) entre jeux classiques :

                  Prisoner Stag Hun Battle o  Chicken Matching 
-----------------------------------------------------------
Prisoner's Di        0        2        5        2        3 
Stag Hunt            2        0        3        4        3 
Battle of Sex        5        3        0        7        4 
Chicken              2        4        7        0        3 
Matching Penn        3        3        4        3        0 


La diagonale = 0 (meme jeu). Symetrie par construction (graphe non oriente).

## 10. Exercices

Trois extensions à compléter (stubs sans erreur — règle C.1). Le notebook s'exécute de bout en bout même non complété.


### Exercice 1 — Voisinage de swap d'un jeu

Étant donné un jeu, lister ses **6 voisins** (3 swaps × 2 joueurs) et classifier chacun (nom de famille + nombre de Nash).

# Indice : itérer sur les 3 paires de swaps, appliquer `ApplyRowSwap` et `ApplyColSwap`, puis `FindPureNash` sur chaque voisin.


In [10]:
// Cellule 10 — Exercice 1 (a completer) : voisinage de swap
// TODO etudiant : implementer ExploreSwapNeighbors(game) -> liste de voisins classes
// Etape 1 : iterer sur swaps [(1,2), (2,3), (3,4)].
// Etape 2 : pour chaque swap, calculer le voisin Row et le voisin Col.
// Etape 3 : pour chaque voisin, compter les Nash purs via FindPureNash.
public List<(OrdinalGame neighbor, string swap, int nashCount)> ExploreSwapNeighbors(OrdinalGame g)
{
    // TODO etudiant
    return new List<(OrdinalGame, string, int)>();   // stub : retourner la liste des 6 voisiers
}

display("Exercice 1 (voisinage de swap) : stub a completer.");


Exercice 1 (voisinage de swap) : stub a completer.

### Exercice 2 — Jeux sans équilibre pur

Combien des 576 jeux n'ont **aucun** Nash pur ? Lesquels sont « proches » de Matching Pennies (distance ≤ 2) ?

# Indice : filtrer `allGames` par `FindPureNash(g).Count == 0`, puis `SwapDistance` vs Matching Pennies.


In [11]:
// Cellule 11 — Exercice 2 (a completer) : jeux sans Nash pur
// TODO etudiant : denombrer les jeux sans Nash pur et identifier les proches de Matching Pennies
// Etape 1 : filtrer allGames pour FindPureNash == 0.
// Etape 2 : calculer SwapDistance vs Matching Pennies pour chaque jeu sans Nash.
public int CountNoNashGames()
{
    // TODO etudiant
    return 0;   // stub : retourner le compte
}

display("Exercice 2 (jeux sans Nash) : stub a completer.");


Exercice 2 (jeux sans Nash) : stub a completer.

### Exercice 3 — Équivalence par renommage des joueurs

Deux jeux sont **stratégiquement équivalents** si l'échange Ligne↔Colonne (transposée) donne le même jeu. Compter les classes d'équivalence des 576 jeux sous cette relation.

# Indice : pour chaque jeu, calculer sa transposée (échanger rôles + transposer matrices), regrouper par `HashSet` de représentants canoniques.


In [12]:
// Cellule 12 — Exercice 3 (a completer) : equivalence par renommage
// TODO etudiant : compter les classes d'equivalence sous echange Ligne<->Colonne
// Etape 1 : definir la transposee (swap des roles : Row<->Col + reindexation des cellules).
// Etape 2 : regrouper les 576 jeux en classes (un representant canonique par classe).
public int CountPlayerSwapClasses()
{
    // TODO etudiant
    return 0;   // stub : retourner le nombre de classes
}

display("Exercice 3 (equivalence renommage) : stub a completer.");


Exercice 3 (equivalence renommage) : stub a completer.

## 11. Résumé

Ce twin C# a reconstruit from-scratch (BCL .NET 9, 0 NuGet) toute la topologie des jeux 2×2 ordinaux :

1. **Représentation ordinale** `OrdinalGame` (rangs 1-4 par joueur, permutation validée).
2. **Swaps de rangs** (transformations élémentaires : 1↔2, 2↔3, 3↔4, par joueur).
3. **BFS swap-path** (shortest-path entre deux jeux — remplace `networkx.shortest_path_length`).
4. **Énumération exhaustive** des 576 jeux (Heap's algorithm for permutations).
5. **Équilibres de Nash purs** (best-response sur les 4 cellules).
6. **Classification** des 576 jeux par nombre de Nash.
7. **Connexité** du graphe des swaps (BFS flood-fill — confirmée : 576/576 atteignables).

### Leçons clés

- **Invariance ordinale** : la structure stratégique (nombre/type de Nash) ne dépend que de l'ordre des gains, pas de leurs valeurs. Le Dilemme du Prisonnier est un dilemme quelle que soit l'échelle.
- **Espace fini explorable** : 576 jeux, graphe connexe — on peut classifier exhaustivement, ce qui est impossible en cardinal.
- **Topologie des swaps** : les jeux classiques forment un « paysage » où la distance mesure la proximité stratégique. Matching Pennies (0 Nash) est loin des jeux de coordination (2 Nash).

### Référence SOTA

Le twin Python [GameTheory-3-Topology2x2](GameTheory-3-Topology2x2.ipynb) résout le même problème avec **networkx** (graphe + BFS + layout), **numpy** (matrices) et **matplotlib** (visualisation en labyrinthe coloré). Ces libs optimisent la manipulation ; le twin C# rend **explicites** les algorithmes sous-jacents (BFS manuel, Heap, best-response). En production : networkx. En pédagogie : comprendre chaque brique, from-scratch.

---

**Marathon #4956** — twin C# .NET ⇄ Python. Suite de [GameTheory-2-NormalForm-Csharp](GameTheory-2-NormalForm-Csharp.ipynb). Voir #4956, #3801. Revue souhaitée : ai-01, po-2024 (GameTheory/.NET owner), po-2025.
